# 🔐 Six-State QKD Protocol – Interactive Demo

This notebook provides a step-by-step walkthrough of the Six-State Quantum Key Distribution protocol, with comparisons to BB84.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np

from quantum.utils import Basis, SIX_STATE_BASES, BB84_BASES
from quantum.alice import Alice
from quantum.bob import Bob
from quantum.eve import Eve
from protocols.six_state import SixStateProtocol
from protocols.bb84 import BB84Protocol
from protocols.base import sift_keys
from analysis.qber import compute_qber
from analysis.key_rate import compute_secure_key_rate, compute_effective_key_rate

print('All imports successful ✅')

## 1. Basic Protocol Run

Let's run the Six-State protocol with default parameters — no Eve, no noise.

In [ ]:
proto = SixStateProtocol(alice_seed=42, bob_seed=43)
result = proto.run(n_qubits=500)

print(f'Qubits transmitted:  {result.n_qubits}')
print(f'Sifted key length:   {result.sifted_length}')
print(f'Sifting rate:        {result.raw_key_rate:.4f}')
print(f'QBER:                {result.qber:.4f} ({result.qber*100:.2f}%)')
print(f'Final key length:    {len(result.final_key)}')
print(f'Eve detected:        {result.eve_detected}')

## 2. Eavesdropping Detection

Now let's enable Eve and observe the QBER increase.

In [ ]:
result_eve = SixStateProtocol(alice_seed=42, bob_seed=43, eve_seed=44).run(
    n_qubits=500, eve_present=True
)

print(f'QBER (no Eve):  {result.qber:.4f}')
print(f'QBER (with Eve): {result_eve.qber:.4f}')
print(f'\nEve detected: {result_eve.eve_detected} ⚠️' if result_eve.eve_detected else f'\nEve NOT detected')

## 3. Six-State vs BB84 Comparison

In [ ]:
n = 1000

six_no_eve = SixStateProtocol().run(n)
six_eve = SixStateProtocol().run(n, eve_present=True)
bb84_no_eve = BB84Protocol().run(n)
bb84_eve = BB84Protocol().run(n, eve_present=True)

print(f'{"Protocol":<12} {"QBER (no Eve)":>15} {"QBER (Eve)":>12} {"Sifting":>10}')
print('-' * 52)
print(f'{"Six-State":<12} {six_no_eve.qber:>15.4f} {six_eve.qber:>12.4f} {six_no_eve.raw_key_rate:>10.4f}')
print(f'{"BB84":<12} {bb84_no_eve.qber:>15.4f} {bb84_eve.qber:>12.4f} {bb84_no_eve.raw_key_rate:>10.4f}')

## 4. QBER vs Eve (Bar Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

protocols = ['Six-State', 'BB84']
x = np.arange(len(protocols))
width = 0.35

bars1 = ax.bar(x - width/2, [six_no_eve.qber, bb84_no_eve.qber], width,
               label='No Eve', color='#4CAF50', alpha=0.85)
bars2 = ax.bar(x + width/2, [six_eve.qber, bb84_eve.qber], width,
               label='With Eve', color='#F44336', alpha=0.85)

ax.set_ylabel('QBER')
ax.set_title('QBER: No Eve vs Eavesdropping')
ax.set_xticks(x)
ax.set_xticklabels(protocols)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5. Noise Sweep – QBER vs Channel Noise

In [ ]:
noise_levels = [0.0, 0.01, 0.02, 0.05, 0.08, 0.1, 0.15, 0.2, 0.3]
six_qbers = []
bb84_qbers = []

for p in noise_levels:
    r = SixStateProtocol().run(500, noise_type='depolarizing', noise_prob=p)
    six_qbers.append(r.qber)
    r = BB84Protocol().run(500, noise_type='depolarizing', noise_prob=p)
    bb84_qbers.append(r.qber)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(noise_levels, six_qbers, 'o-', color='#2196F3', linewidth=2, label='Six-State')
ax.plot(noise_levels, bb84_qbers, 's--', color='#FF9800', linewidth=2, label='BB84')
ax.axhline(y=1/3, color='#2196F3', linestyle=':', alpha=0.5, label='Six-State threshold')
ax.axhline(y=0.25, color='#FF9800', linestyle=':', alpha=0.5, label='BB84 threshold')
ax.set_xlabel('Noise Probability')
ax.set_ylabel('QBER')
ax.set_title('QBER vs Depolarizing Noise')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Key Rate vs Number of Qubits

In [ ]:
qubit_counts = [50, 100, 200, 500, 750, 1000, 1500, 2000]
six_rates = []
bb84_rates = []

for n in qubit_counts:
    r = SixStateProtocol().run(n)
    six_rates.append(compute_effective_key_rate(n, r.sifted_length, r.qber, 'six_state'))
    r = BB84Protocol().run(n)
    bb84_rates.append(compute_effective_key_rate(n, r.sifted_length, r.qber, 'bb84'))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(qubit_counts, six_rates, 'o-', color='#2196F3', linewidth=2, label='Six-State')
ax.plot(qubit_counts, bb84_rates, 's--', color='#FF9800', linewidth=2, label='BB84')
ax.set_xlabel('Number of Qubits')
ax.set_ylabel('Effective Key Rate (bits/qubit)')
ax.set_title('Key Rate vs Number of Qubits')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Secure Key Rate vs QBER (Theoretical)

In [ ]:
qbers = np.linspace(0, 0.5, 200)
six_rates_th = [compute_secure_key_rate(q, 'six_state') for q in qbers]
bb84_rates_th = [compute_secure_key_rate(q, 'bb84') for q in qbers]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(qbers, six_rates_th, '-', color='#2196F3', linewidth=2, label='Six-State')
ax.plot(qbers, bb84_rates_th, '--', color='#FF9800', linewidth=2, label='BB84')
ax.axvline(x=1/3, color='#2196F3', linestyle=':', alpha=0.5)
ax.axvline(x=0.25, color='#FF9800', linestyle=':', alpha=0.5)
ax.fill_between(qbers, 0, six_rates_th, alpha=0.07, color='#2196F3')
ax.fill_between(qbers, 0, bb84_rates_th, alpha=0.07, color='#FF9800')
ax.set_xlabel('QBER')
ax.set_ylabel('Secure Key Rate (bits/sifted bit)')
ax.set_title('Theoretical Secure Key Rate vs QBER')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.5)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 8. Step-by-Step Protocol Walkthrough

Let's look under the hood at individual qubit preparation and measurement.

In [ ]:
# Create Alice and prepare a small number of qubits
alice = Alice(basis_set=SIX_STATE_BASES, seed=100)
state = alice.prepare_qubits(10)

print('Alice\'s preparation:')
print(f'{"Qubit":<8} {"Bit":<6} {"Basis":<8}')
print('-' * 22)
for i in range(10):
    print(f'{i:<8} {state.bits[i]:<6} {state.bases[i].value:<8}')

print(f'\nExample circuit (qubit 0, bit={state.bits[0]}, basis={state.bases[0].value}):')
print(state.circuits[0].draw())

In [ ]:
# Bob measures
bob = Bob(basis_set=SIX_STATE_BASES, seed=101)
bob_state = bob.measure_qubits(state.circuits)

print('\nBasis reconciliation:')
print(f'{"Qubit":<8} {"Alice Basis":<13} {"Bob Basis":<11} {"Match":<8} {"A bit":<7} {"B bit":<7} {"OK":<5}')
print('-' * 60)
for i in range(10):
    match = state.bases[i] == bob_state.bases[i]
    ok = state.bits[i] == bob_state.results[i] if match else '-'
    print(f'{i:<8} {state.bases[i].value:<13} {bob_state.bases[i].value:<11} {str(match):<8} {state.bits[i]:<7} {bob_state.results[i]:<7} {str(ok):<5}')

## Summary

This demo demonstrated:
- ✅ Qubit preparation in Z, X, Y bases using Qiskit circuits
- ✅ Basis reconciliation (sifting)
- ✅ QBER computation with and without Eve
- ✅ Intercept-resend attack detection
- ✅ Noise channel effects
- ✅ Six-State vs BB84 comparison
- ✅ Key rate analysis